# 03 | TNTensor and Compute Backends

This notebook explains why the project wraps raw framework tensors in `TNTensor`, and what the PyTorch/JAX backend abstraction provides.

The key ideas are effective values, scale tracking, reference views, batch axes, and gradient forwarding.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
import torch
from tneq_qc import BackendFactory, TNTensor

backend = BackendFactory.create_backend("pytorch", device="cpu", dtype="float32")


## 1. Effective value: `tensor × scale`

`TNTensor(raw, scale=s)` represents the mathematical value `raw * s`. During deep contractions, the framework can normalize the raw tensor and move its magnitude into `scale`, reducing overflow and underflow risk.

`.tensor` exposes the unscaled body. User-facing conversions such as `.numpy()` and `.item()` include the scale.


In [3]:
raw = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
x = TNTensor(raw, scale=10.0)

print("Raw body:\n", x.tensor)
print("Scale:", x.scale)
print("Effective value:\n", x.numpy())


Raw body:
 tensor([[1., 2.],
        [3., 4.]])
Scale: 10.0
Effective value:
 [[10. 20.]
 [30. 40.]]


In [4]:
y = TNTensor(torch.tensor([[1000.0, -500.0]]))
before = y.numpy().copy()
y.auto_scale()

print(y)
print("Effective value preserved:", (before == y.numpy()).all())


TNTensor(shape=torch.Size([1, 2]), scale=1000)
Effective value preserved: True


## 2. Arithmetic propagates scale

`TNTensor` implements arithmetic, matrix multiplication, reshape, transpose, reductions, and einsum. The result remains a `TNTensor`, and scale is propagated according to the mathematical operation.


In [5]:
a = TNTensor(torch.tensor([[1.0, 2.0], [3.0, 4.0]]), scale=2.0)
b = TNTensor(torch.eye(2), scale=3.0)

product = a @ b
total = a + b

print("Product scale:", product.scale)
print("Product effective value:\n", product.numpy())
print("Sum effective value:\n", total.numpy())


Product scale: 6.0
Product effective value:
 [[ 6. 12.]
 [18. 24.]]
Sum effective value:
 [[ 5.  4.]
 [ 6. 11.]]


## 3. Gradient forwarding

`requires_grad_()`, `grad`, `is_leaf`, and `backward()` delegate to the underlying tensor. Optimizers can therefore work with `TNTensor` while PyTorch still performs autodiff.


In [6]:
parameter = TNTensor(torch.tensor([1.0, -2.0]))
parameter.requires_grad_(True)

loss = (parameter.tensor ** 2).sum()
loss.backward()

print("requires_grad:", parameter.requires_grad)
print("is_leaf:", parameter.is_leaf)
print("gradient:", parameter.grad)


requires_grad: True
is_leaf: True
gradient: tensor([ 2., -4.])


## 4. Hermitian reference views

`hermit()` and `conj_transpose()` return reference views rather than independent trainable parameters. They conjugate and permute the source tensor while preserving the autograd link back to the source.

This is how a BornMachine shares parameters between `TN` and `TN†`.


In [7]:
complex_raw = torch.tensor(
    [[1 + 2j, 3 - 1j], [0 + 1j, 2 + 0j]],
    dtype=torch.complex64,
)
z = TNTensor(complex_raw)
z_h = z.hermit()

print("Reference view:", z_h.is_ref)
print("Original:\n", z.numpy())
print("Hermitian:\n", z_h.numpy())


Reference view: True
Original:
 [[1.+2.j 3.-1.j]
 [0.+1.j 2.+0.j]]
Hermitian:
 [[1.-2.j 0.-1.j]
 [3.+1.j 2.+0.j]]


## 5. Batch axes

`has_batch=True` means axis 0 enumerates samples rather than representing a tensor-network edge. Contraction strategies preserve and align this axis.


In [8]:
batched = TNTensor(torch.randn(8, 2, 2), has_batch=True)
print("Shape:", batched.shape)
print("Has batch:", batched.has_batch)


Shape: torch.Size([8, 2, 2])
Has batch: True


## 6. Backend responsibilities and automatic scaling

A backend unifies tensor creation, einsum, gradients, devices, and dtypes. Enabling automatic scaling allows supported contraction paths to normalize intermediate `TNTensor` results.


In [9]:
stable_backend = BackendFactory.create_backend(
    "pytorch",
    device="cpu",
    dtype="complex64",
    enable_auto_scale=True,
)
print(stable_backend.get_backend_info())


BackendInfo(backend_type='pytorch', device='cpu', dtype='complex64', config={'enable_auto_scale': True})


### Practical choices

- Start with PyTorch, CPU, and `float32`.
- Use `complex64` for amplitudes and conjugate operations.
- Try `enable_auto_scale=True` if deep contractions become numerically unstable.
- The JAX backend exists, but the repository's examples and tests focus more heavily on PyTorch.

### Exercises

- Compare `clone()` and `hermit()` after changing the source tensor.
- Add and multiply tensors with different scales and verify effective values manually.
- Create a batch of four matrices and experiment with `sum()` and `mean()`.
